In [1]:
import sqlite3, pandas as pd
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
con = sqlite3.connect(":memory:")
df.to_sql("customers", con, index=False)


7043

In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Clean TotalCharges & separate X / y
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["MonthlyCharges"] * df["tenure"])

X = df.drop(columns=["customerID", "Churn", "tenure_group"], errors="ignore")
y = (
    df["Churn"].map({"Yes": 1, "No": 0}).astype(int)
    if pd.api.types.is_string_dtype(df["Churn"])
    else df["Churn"].astype(int)
)

# 2. Stratified 80/20 Train/Test Split (Test set stays untouched during CV)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Shared Leak-Proof ColumnTransformer
# sparse_output=False ensures compatibility with HistGradientBoostingClassifier
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            cat_cols,
        ),
    ]
)

# 4. Define All Three Pipelines
pipelines = {
    "Dummy Classifier": Pipeline([
        ("preprocessor", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent", random_state=42)),
    ]),
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    "HistGradientBoosting": Pipeline([
        ("preprocessor", preprocessor),
        ("model", HistGradientBoostingClassifier(random_state=42)),
    ]),
}

# 5. 5-Fold Stratified Cross-Validation on Training Set ONLY
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
for name, pipe in pipelines.items():
    scores = cross_val_score(
        pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
    )
    cv_results.append({
        "Model": name,
        "Mean ROC AUC": scores.mean(),
        "Std Dev (Spread)": scores.std(),
        "Min Fold": scores.min(),
        "Max Fold": scores.max(),
    })

results_df = pd.DataFrame(cv_results).sort_values(by="Mean ROC AUC", ascending=False)
print("=== 5-Fold Stratified Cross-Validation (Training Set) ===")
print(results_df.to_string(index=False, float_format="%.4f"))

# 6. Pick the Winner Based on CV Mean ROC AUC & Evaluate ONCE on Test Set
winner_name = results_df.iloc[0]["Model"]
winner_pipe = pipelines[winner_name]

winner_pipe.fit(X_train, y_train)
y_test_probs = winner_pipe.predict_proba(X_test)[:, 1]

test_auc = roc_auc_score(y_test, y_test_probs)
k = int(len(y_test) * 0.20)
top_20_idx = np.argsort(y_test_probs)[::-1][:k]
test_prec_20 = y_test.iloc[top_20_idx].mean()

print(f"\n=== Final Hold-Out Test Evaluation: {winner_name} ===")
print(f"Test ROC AUC:             {test_auc:.4f}")
print(f"Test Precision @ Top 20%: {test_prec_20:.4f} ({test_prec_20*100:.1f}%)")


=== 5-Fold Stratified Cross-Validation (Training Set) ===
               Model  Mean ROC AUC  Std Dev (Spread)  Min Fold  Max Fold
 Logistic Regression        0.8462            0.0126    0.8252    0.8630
HistGradientBoosting        0.8364            0.0054    0.8261    0.8420
    Dummy Classifier        0.5000            0.0000    0.5000    0.5000

=== Final Hold-Out Test Evaluation: Logistic Regression ===
Test ROC AUC:             0.8421
Test Precision @ Top 20%: 0.6619 (66.2%)
